In [1]:
print("Hello from Python. My Week 5 notebook is running.")

Hello from Python. My Week 5 notebook is running.


In [2]:
import os
os.getcwd()

'/workspaces/HDSx-workspace/assignments/assignment04-polyglot/submission'

In [3]:
import pandas as pd

nhanes = pd.read_csv(
    "../../../examples/nhanes-equity/data/nhanes_equity_v6.csv",
    low_memory=False
)

In [4]:
nhanes.head()

,Cycle,BMI,Weight,Height,Waist,WeightMEC,Strata,PSU,Gender,Race,Education,Marital,Age,PIR,EducationClean,MaritalClean,IncomeGroup,WHtR
0,1999-2000,14.90,12.5,91.6,45.7,10982.898896,5,1,Female,Non-Hispanic Black,NaN,NaN,2,0.86,NaN,NaN,Low Income (<1.3),0.498908
1,1999-2000,24.90,75.4,174.0,98.0,28325.384898,1,3,Male,Non-Hispanic White,College Grad,NaN,77,5.00,College Grad,NaN,High Income (>3.5),0.563218
2,1999-2000,17.63,32.9,136.6,64.7,46192.256945,7,2,Female,Non-Hispanic White,NaN,NaN,10,1.47,NaN,NaN,Middle Income,0.473646
3,1999-2000,NaN,13.3,NaN,NaN,10251.260020,2,1,Male,Non-Hispanic Black,NaN,NaN,1,0.57,NaN,NaN,Low Income (<1.3),NaN
4,1999-2000,29.10,92.5,178.3,99.9,99445.065735,8,2,Male,Non-Hispanic White,College Grad,Married/Partner,49,5.00,College Grad,Married/Partner,High Income (>3.5),0.560292


In [5]:
nhanes.shape

(105626, 18)

Audit check: the Python dataset has 105626 rows and 18 columns, which matches the R dataset.

In [6]:
nhanes.columns.tolist()

['Cycle',
 'BMI',
 'Weight',
 'Height',
 'Waist',
 'WeightMEC',
 'Strata',
 'PSU',
 'Gender',
 'Race',
 'Education',
 'Marital',
 'Age',
 'PIR',
 'EducationClean',
 'MaritalClean',
 'IncomeGroup',
 'WHtR']

In [7]:
nhanes[["BMI", "IncomeGroup", "Gender"]].isna().sum()

BMI            9356
IncomeGroup    9415
Gender            0
dtype: int64

Audit check: Python has 9356 missing BMI values, which matches the R result. Missing values are checked before creating the grouped summary.

In [8]:
python_summary = (
    nhanes
    .dropna(subset=["BMI", "IncomeGroup", "Gender"])
    .groupby(["IncomeGroup", "Gender"], as_index=False)
    .agg(
        n=("BMI", "size"),
        mean_bmi=("BMI", "mean"),
        sd_bmi=("BMI", "std")
    )
    .sort_values(["IncomeGroup", "Gender"])
)

python_summary[["mean_bmi", "sd_bmi"]] = (
    python_summary[["mean_bmi", "sd_bmi"]].round(2)
)

python_summary

,IncomeGroup,Gender,n,mean_bmi,sd_bmi
0,High Income (>3.5),Female,11743,25.74,7.51
1,High Income (>3.5),Male,12095,26.11,6.81
2,Low Income (<1.3),Female,16548,25.95,8.65
3,Low Income (<1.3),Male,14847,24.05,7.17
4,Middle Income,Female,16432,26.39,8.20
5,Middle Income,Male,15982,25.52,7.22


Audit check: the Python summary produces the same six IncomeGroup and Gender groups as the R summary, and the group counts, mean BMI values, and standard deviations match after rounding to two decimal places.
    

In [9]:
from helpers import summarise_mean_sd

python_function_summary = summarise_mean_sd(
    nhanes,
    "BMI",
    "IncomeGroup",
    "Gender"
)

python_function_summary

,IncomeGroup,Gender,n,mean_value,sd_value
0,High Income (>3.5),Female,11743,25.74,7.51
1,High Income (>3.5),Male,12095,26.11,6.81
2,Low Income (<1.3),Female,16548,25.95,8.65
3,Low Income (<1.3),Male,14847,24.05,7.17
4,Middle Income,Female,16432,26.39,8.20
5,Middle Income,Male,15982,25.52,7.22


In [10]:
filtered_python = nhanes.dropna(
    subset=["BMI", "IncomeGroup", "Gender"]
)

len(nhanes), len(filtered_python)

(105626, 87647)

In [11]:
nhanes.shape[1], len(python_summary)

(18, 6)

In [12]:
parity_table = pd.DataFrame({
    "metric": [
        "raw_row_count",
        "raw_column_count",
        "missing_bmi_count",
        "filtered_row_count",
        "number_of_groups",
        "grouping_labels",
        "rounded_summary_values"
    ],
    "r_value": [
        105626,
        18,
        9356,
        87647,
        6,
        "High Income (>3.5) Female; High Income (>3.5) Male; Low Income (<1.3) Female; Low Income (<1.3) Male; Middle Income Female; Middle Income Male",
        "High Female: n=11743, mean=25.74, SD=7.51; High Male: n=12095, mean=26.11, SD=6.81; Low Female: n=16548, mean=25.95, SD=8.65; Low Male: n=14847, mean=24.05, SD=7.17; Middle Female: n=16432, mean=26.39, SD=8.20; Middle Male: n=15982, mean=25.52, SD=7.22"
    ],
    "python_value": [
        105626,
        18,
        9356,
        87647,
        6,
        "High Income (>3.5) Female; High Income (>3.5) Male; Low Income (<1.3) Female; Low Income (<1.3) Male; Middle Income Female; Middle Income Male",
        "High Female: n=11743, mean=25.74, SD=7.51; High Male: n=12095, mean=26.11, SD=6.81; Low Female: n=16548, mean=25.95, SD=8.65; Low Male: n=14847, mean=24.05, SD=7.17; Middle Female: n=16432, mean=26.39, SD=8.20; Middle Male: n=15982, mean=25.52, SD=7.22"
    ],
    "match": [
        True,
        True,
        True,
        True,
        True,
        True,
        True
    ],
    "notes": [
        "",
        "",
        "Both languages have 9356 missing BMI values",
        "",
        "",
        "IncomeGroup and Gender labels match in the same order",
        "IncomeGroup/Gender counts, means, and SDs agree after rounding to 2 decimals"
    ]
})

parity_table

,metric,r_value,python_value,match,notes
0,raw_row_count,105626,105626,True,
1,raw_column_count,18,18,True,
2,missing_bmi_count,9356,9356,True,Both languages have 9356 missing BMI values
3,filtered_row_count,87647,87647,True,
4,number_of_groups,6,6,True,
5,grouping_labels,High Income (>3.5) Female; High Income (>3.5) ...,High Income (>3.5) Female; High Income (>3.5) ...,True,IncomeGroup and Gender labels match in the sam...
6,rounded_summary_values,"High Female: n=11743, mean=25.74, SD=7.51; Hig...","High Female: n=11743, mean=25.74, SD=7.51; Hig...",True,"IncomeGroup/Gender counts, means, and SDs agre..."


In [13]:
parity_table.to_csv("parity-table.csv", index=False)

In [14]:
nhanes["BMI"].isna().sum()

np.int64(9356)

In [15]:
python_summary[["IncomeGroup", "Gender"]]

,IncomeGroup,Gender
0,High Income (>3.5),Female
1,High Income (>3.5),Male
2,Low Income (<1.3),Female
3,Low Income (<1.3),Male
4,Middle Income,Female
5,Middle Income,Male


### AI Prompt Used

How do I translate my R BMI summary by IncomeGroup and Gender into Python/pandas?

## AI Use and Verification

I used AI to help translate the R workflow into Python/pandas and troubleshoot Jupyter notebook issues. I verified the translated workflow by comparing the R and Python row counts, missing BMI values, grouping labels, and rounded BMI summary values.